In [5]:
# ==========================================
# 🧠 AI TRAINING ON COLAB (Final Corrected)
# ==========================================

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from sklearn.model_selection import train_test_split
from google.colab import drive
import pickle
import os

# 1. MOUNT DRIVE
print("📂 Mounting Google Drive...")
drive.mount('/content/drive')

# ==========================================
# 🔍 AUTO-DETECT FILE (Jasoos Mode)
# ==========================================
TARGET_FILE_NAME = "final_dataset_ULTIMATE.parquet"
INPUT_FILE = None

print(f"\n🕵️‍♂️ Searching for '{TARGET_FILE_NAME}' in Google Drive...")

# Poori Drive mein dhundega (Auto-Search)
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if TARGET_FILE_NAME in files:
        INPUT_FILE = os.path.join(root, TARGET_FILE_NAME)
        print(f"✅ FILE MIL GAYI! (FOUND): {INPUT_FILE}")
        break

if INPUT_FILE is None:
    print(f"\n❌ Error: '{TARGET_FILE_NAME}' puri Drive mein kahin nahi mili!")
    print("👉 Check karo ki file upload ho gayi hai ya nahi.")
    raise SystemExit("File not found.")

# Output Paths (Wahin save karenge jahan file mili)
SAVE_DIR = os.path.dirname(INPUT_FILE)
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'phishguard_model_cnn.h5')
TOKENIZER_SAVE_PATH = os.path.join(SAVE_DIR, 'tokenizer.pkl')

# ==========================================
# ⚙️ MODEL SETTINGS
# ==========================================
MAX_WORDS = 200000
MAX_LEN = 150
BATCH_SIZE = 1024
EPOCHS = 5

# ==========================================
# 🚀 TRAINING ENGINE
# ==========================================
def start_training():
    print(f"\n🚀 Starting Training on GPU: {tf.test.gpu_device_name()}")

    # 1. LOAD DATA
    print("⏳ Loading Parquet Data...")
    df = pd.read_parquet(INPUT_FILE)
    print(f"   ✅ Data Loaded: {len(df)} Rows")

    urls = df['url'].astype(str).values
    labels = df['label'].values

    # 2. TOKENIZATION
    print("🔤 Tokenizing...")
    tokenizer = Tokenizer(num_words=MAX_WORDS, char_level=True, oov_token='<OOV>')
    tokenizer.fit_on_texts(urls)

    # Save Tokenizer
    print(f"   💾 Saving Tokenizer to: {TOKENIZER_SAVE_PATH}")
    with open(TOKENIZER_SAVE_PATH, 'wb') as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

    sequences = tokenizer.texts_to_sequences(urls)
    X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

    # Free RAM
    del df, urls, sequences

    # 3. SPLIT
    print("✂️ Splitting Data...")
    X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)

    # 4. BUILD CNN MODEL
    print("🛠️ Building CNN Model...")
    model = Sequential([
        Embedding(MAX_WORDS, 32, input_length=MAX_LEN),
        Conv1D(64, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    # 5. TRAIN
    print(f"\n🔥 TRAINING STARTED ({EPOCHS} Epochs)...")
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )

    # 6. SAVE MODEL
    print(f"\n💾 Saving Model to: {MODEL_SAVE_PATH}")
    model.save(MODEL_SAVE_PATH)
    print("🎉 MISSION SUCCESS! Model aur Tokenizer wahi folder mein save ho gaye hain.")

if __name__ == "__main__":
    start_training()

📂 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🕵️‍♂️ Searching for 'final_dataset_ULTIMATE.parquet' in Google Drive...
✅ FILE MIL GAYI! (FOUND): /content/drive/MyDrive/Colab Notebooks/Dataset/final_dataset_ULTIMATE.parquet

🚀 Starting Training on GPU: 
⏳ Loading Parquet Data...
   ✅ Data Loaded: 4826309 Rows
🔤 Tokenizing...
   💾 Saving Tokenizer to: /content/drive/MyDrive/Colab Notebooks/Dataset/tokenizer.pkl
✂️ Splitting Data...
🛠️ Building CNN Model...

🔥 TRAINING STARTED (5 Epochs)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
3771/3771 ━━━━━━━━━━━━━━━━━━━━ 1297s 343ms/step - accuracy: 0.9322 - loss: 0.1689 - val_accuracy: 0.9534 - val_loss: 0.1104
Epoch 2/5
3771/3771 ━━━━━━━━━━━━━━━━━━━━ 1272s 337ms/step - accuracy: 0.9558 - loss: 0.1092 - val_accuracy: 0.9586 - val_loss: 0.1012
Epoch 3/5
3771/3771 ━━━━━━━━━━━━━━━━━━━━ 1309s 344ms/step - accuracy: 0.9586 - loss: 0.1024 - val_accuracy: 0.9603 - val_loss: 0.0972
Epoch 4/5
3771/3771 ━━━━━━━━━━━━━━━━━━━━ 1300s 345ms/step - accuracy: 0.9606 - loss: 0.0985 - val_accuracy: 0.9614 - val_loss: 0.0939
Epoch 5/5
3771/3771 ━━━━━━━━━━━━━━━━━━━━ 1327s 341ms/step - accuracy: 0.9615 - loss: 0.0959 - val_accuracy: 0.9618 - val_loss: 0.0938



💾 Saving Model to: /content/drive/MyDrive/Colab Notebooks/Dataset/phishguard_model_cnn.h5
🎉 MISSION SUCCESS! Model aur Tokenizer wahi folder mein save ho gaye hain.
